# 06 — 記憶與 Checkpointer：讓 graph 記得「之前發生過的事」

> 不需要 API key。這章講的是 LangGraph 怎麼「存檔」，跟用哪個 LLM 無關。

到目前為止，每次 `graph.invoke(...)` 都是各自獨立的一次執行——執行完就忘光，下一次呼叫等於重新開機。這一份要解決的問題是：**怎麼讓圖記得上一次對話說過什麼**。

用玩遊戲比喻：

- **Checkpointer**：遊戲的「自動存檔系統」。每走一步（每個 node 執行完）就偷偷幫你存一次檔，完全不用自己按存檔鍵。這裡用的 `InMemorySaver` 是最簡單的一種，把存檔資料放在程式的記憶體裡。
- **`thread_id`**：你的「存檔檔案」。同一個 `thread_id` 就是接著同一個存檔繼續玩；換一個 `thread_id` 就是開新的一局，兩邊互不干擾。

具體做法只差一行：`builder.compile(checkpointer=InMemorySaver())`。之後呼叫時帶上同一個 `thread_id`，狀態就會自動接續，不用自己手動把歷史訊息組進輸入。

先看一眼等等會發生的事（對照下面 code cell 的真實輸出）：

```
thread_id="session-1"          ← 一個存檔檔案
  invoke("第一句話")
    存檔內容: [第一句話, 目前對話共 2 則]
  invoke("第二句話")            ← 沒有重傳第一句話，是自動接上去的
    存檔內容: [第一句話, 目前對話共 2 則, 第二句話, 目前對話共 4 則]

thread_id="session-2"          ← 換一個存檔檔案 = 全新一局
  invoke("這是另一個對話")
    存檔內容: [這是另一個對話, 目前對話共 2 則]   ← 跟 session-1 完全無關
```

In [1]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph


def echo(state: MessagesState) -> dict:
    return {"messages": [("ai", f"目前對話共 {len(state['messages']) + 1} 則")]}


builder = StateGraph(MessagesState)
builder.add_node("echo", echo)
builder.add_edge(START, "echo")
builder.add_edge("echo", END)

graph = builder.compile(checkpointer=InMemorySaver())

## `thread_id`：呼叫時用它指定「要接續哪個存檔」

透過 `config={"configurable": {"thread_id": ...}}` 告訴 LangGraph 要接的是哪一局。用同一個
`thread_id` 呼叫兩次，第二次的輸入會自動接在第一次的狀態後面——這是因為 `messages` 欄位標了
`add_messages` reducer（見 `03_langgraph_core.ipynb`），新訊息是「累加」而不是「覆蓋」。

In [2]:
config = {"configurable": {"thread_id": "session-1"}}

result_1 = graph.invoke({"messages": [("human", "第一句話")]}, config)
print(result_1["messages"][-1].content)

result_2 = graph.invoke({"messages": [("human", "第二句話")]}, config)
print([m.content for m in result_2["messages"]])  # 累積了四則：兩次 human + 兩次 ai

目前對話共 2 則
['第一句話', '目前對話共 2 則', '第二句話', '目前對話共 4 則']


## 換一個 `thread_id`，就是開新的一局

Checkpointer 是照 `thread_id` 分開存檔的。換一個 id，等於拿了另一個存檔檔案，完全看不到別的
session 存了什麼。

In [3]:
other_config = {"configurable": {"thread_id": "session-2"}}
other_result = graph.invoke({"messages": [("human", "這是另一個對話")]}, other_config)
print([m.content for m in other_result["messages"]])  # 只有兩則，跟 session-1 無關

['這是另一個對話', '目前對話共 2 則']


## `get_state` / `get_state_history`：看目前狀態，或倒轉回去看每一步

`get_state` 是「讀目前這個存檔檔案的最新進度」；`get_state_history` 是「把這個存檔檔案每一步的
快照都列出來」，由新到舊排序（倒序）。

`metadata["step"]` 是這個 `thread_id` 從頭到尾累加的步數——不是每次呼叫都重新從 0 算，而是同一個
存檔一路往上跳。`-1` 是「這個 thread 還沒套用任何輸入前」的起點，之後每呼叫一次圖，這個數字就繼續往上加。

In [4]:
print("目前狀態:", [m.content for m in graph.get_state(config).values["messages"]])
print()
print("歷史快照（倒序）:")
for snapshot in graph.get_state_history(config):
    print(" step", snapshot.metadata.get("step"), "->", [m.content for m in snapshot.values["messages"]])

目前狀態: ['第一句話', '目前對話共 2 則', '第二句話', '目前對話共 4 則']

歷史快照（倒序）:
 step 4 -> ['第一句話', '目前對話共 2 則', '第二句話', '目前對話共 4 則']
 step 3 -> ['第一句話', '目前對話共 2 則', '第二句話']
 step 2 -> ['第一句話', '目前對話共 2 則']
 step 1 -> ['第一句話', '目前對話共 2 則']
 step 0 -> ['第一句話']
 step -1 -> []


## `update_state`：不執行圖，直接改存檔內容

有時候需要「人工修正」對話歷史（例如審核後修改某一則訊息），不需要重新跑一次圖，直接呼叫
`update_state` 改存檔內容即可——這在 `07_langgraph_human_in_the_loop.ipynb` 的審核流程裡也會
用到類似的機制。

In [5]:
graph.update_state(config, {"messages": [("human", "這則是用 update_state 手動插入的")]})
print([m.content for m in graph.get_state(config).values["messages"]])

['第一句話', '目前對話共 2 則', '第二句話', '目前對話共 4 則', '這則是用 update_state 手動插入的']


## 小結
- Checkpointer 讓「多輪對話」變成「同一個 `thread_id` 反覆呼叫」，狀態自動接續，不用自己手動
  組裝歷史訊息——就像同一個存檔檔案玩到哪存到哪
- `InMemorySaver` 只存在行程記憶體裡，程式一關就消失——正式環境要換成
  `SqliteSaver` / `PostgresSaver`（`10_langgraph_persistence_deploy.ipynb` 會示範）
- `get_state_history` 讓你可以倒轉回去看「圖到底一步一步做了什麼」，`update_state` 讓你可以
  在不重跑的情況下修正存檔內容

下一份：`07_langgraph_human_in_the_loop.ipynb`，用 `interrupt()` 讓圖在某個節點暫停，
等人工核准後再繼續——這一招也是靠這章的 Checkpointer 才能運作。